# **Distilling**

Con este notebook hemos usado el método distilling de la librería para crear un modelo "destilado", teniendo así un décimo modelo predictivo.

**Necesario archivo auxiliar disilling.py**, o en su defecto, tener la librería `glassbox_vit` instalada.


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub import download_bucket_files, list_bucket_tree
from pathlib import Path
import os
from datasets import load_dataset, Image
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification, AutoImageProcessor, default_data_collator
from glassbox_vit.distilling import KnowledgeDistillationTrainer
from sklearn.metrics import classification_report, matthews_corrcoef
import numpy as np

In [ ]:
os.environ["HF_TOKEN"] = "mi_token"

token = os.environ.get("HF_TOKEN")
if not token:
    raise EnvironmentError("Define HF_TOKEN en el entorno antes de ejecutar esta celda")

api = HfApi(token=token)

bucket_id = "z72pepee/tfm-explicabilidad"

carpeta_objetivo = "Modelos_Entrenados/experimento_dataset3_facebook_dinov2/"

# Obtenemos la lista exacta de archivos dentro de esa carpeta
archivos_remotos = [
    item.path
    for item in list_bucket_tree(bucket_id, prefix=carpeta_objetivo)
    if item.type == "file"
]

# Construimos la lista de tuplas que necesita download_bucket_files
# Formato: (ruta_en_el_bucket, ruta_en_tu_disco_local)
# Al poner (ruta, ruta) se guardarán conservando la misma estructura de carpetas.
archivos_a_descargar = [
    (ruta, ruta)
    for ruta in archivos_remotos
]

print(f"¡Se han encontrado {len(archivos_a_descargar)} archivos para descargar!")

# Descargamos los archivos en bloque
if archivos_a_descargar:
    download_bucket_files(
        bucket_id,
        files=archivos_a_descargar
    )
    print("¡Descarga completada con éxito!")
else:
    print("No se encontró nada para descargar (la carpeta no existe o está vacía).")



¡Se han encontrado 12 archivos para descargar!


¡Descarga completada con éxito!


In [ ]:
data3 = load_dataset("z72pepee/skin-gan-review-baseline", "PH2", token="mi_token")

README.md:   0%|          | 0.00/35.4k [00:00<?, ?B/s]

PH2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.6MB            

PH2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.44MB            

PH2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

PH2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.49MB            

PH2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/160 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

# Cargar  modelo profesor
ruta_profesor = "Modelos_Entrenados/experimento_dataset3_facebook_dinov2/checkpoint-30/"
teacher_model = AutoModelForImageClassification.from_pretrained(ruta_profesor)

num_clases = teacher_model.config.num_labels
print(f"El profesor está entrenado para {num_clases} clases.")

# Cargar el modelo estudiante base
student_model = AutoModelForImageClassification.from_pretrained(
    "facebook/dinov2-small-imagenet1k-1-layer",
    num_labels=num_clases,
    ignore_mismatched_sizes=True
)

# Cargar el procesador de imágenes
processor = AutoImageProcessor.from_pretrained(ruta_profesor)

# Definir la función de transformación
def transform_images(examples):
    images = [img.convert("RGB") for img in examples["image"]]
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

# Aplicar la transformación
processed_train_dataset = data3["train"].with_transform(transform_images)
processed_val_dataset = data3["validation"].with_transform(transform_images)

# Crear los DataLoaders
train_loader = DataLoader(
    processed_train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=default_data_collator
)

val_loader = DataLoader(
    processed_val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator
)

# Configurar el Optimizador
optimizer = AdamW(student_model.parameters(), lr=5e-5)

# Instanciar el Trainer
trainer = KnowledgeDistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    temperature=4.0,
    alpha=0.5
)
# Comenzar el entrenamiento
epochs = 10
trainer.train(epochs=epochs)

# Guardar el modelo
save_directory = "./modelo_estudiante_destilado"
trainer.save_student(save_directory)
processor.save_pretrained(save_directory)
print("¡Completado! Modelo estudiante listo para usar")

Entrenando en: cuda


Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

El profesor está entrenado para 2 clases.


config.json:   0%|          | 0.00/58.3k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 91.3MB            

... (2 outputs left out)

[transformers] Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-small-imagenet1k-1-layer
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Starting Knowledge Distillation on cuda for 10 epochs...
Epoch [1/10] | Train Loss: 2.0847 | Val Loss: 1.7166 | Val Accuracy: 85.00%
Epoch [2/10] | Train Loss: 0.6341 | Val Loss: 0.6355 | Val Accuracy: 80.00%
Epoch [3/10] | Train Loss: 0.3420 | Val Loss: 0.3053 | Val Accuracy: 90.00%
Epoch [4/10] | Train Loss: 0.2286 | Val Loss: 0.2662 | Val Accuracy: 85.00%
Epoch [5/10] | Train Loss: 0.1603 | Val Loss: 0.2895 | Val Accuracy: 90.00%
Epoch [6/10] | Train Loss: 0.1558 | Val Loss: 0.2729 | Val Accuracy: 90.00%
Epoch [7/10] | Train Loss: 0.1320 | Val Loss: 0.2185 | Val Accuracy: 90.00%
Epoch [8/10] | Train Loss: 0.1152 | Val Loss: 0.2185 | Val Accuracy: 95.00%
Epoch [9/10] | Train Loss: 0.1104 | Val Loss: 0.2106 | Val Accuracy: 95.00%
Epoch [10/10] | Train Loss: 0.1168 | Val Loss: 0.2127 | Val Accuracy: 95.00%
Training complete! The student has been successfully distilled.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Student model successfully saved to './modelo_estudiante_destilado'
¡Completado! Modelo estudiante listo para usar


In [ ]:
def evaluar_modelo(modelo, dataloader, device, nombre_modelo="Modelo", nombre_dataset="Dataset"):
    """
    Evalúa el modelo y muestra por consola el reporte de clasificación (F1, Precision, Recall) y el MCC.
    """
    print("\n" + "="*60)
    print(f"EVALUACION: {nombre_modelo}")
    print(f"Dataset: {nombre_dataset}")
    print("="*60)

    modelo.eval()
    modelo.to(device)

    todas_las_predicciones = []
    todas_las_etiquetas = []

    with torch.no_grad():
        for batch in dataloader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = modelo(pixel_values=pixel_values)
            logits = outputs.logits

            predicciones = torch.argmax(logits, dim=-1)

            todas_las_predicciones.extend(predicciones.cpu().numpy())
            todas_las_etiquetas.extend(labels.cpu().numpy())

    y_true = np.array(todas_las_etiquetas)
    y_pred = np.array(todas_las_predicciones)

    print("\n--- Reporte de Clasificacion ---")
    print(classification_report(y_true, y_pred, digits=4))

    print("--- Matthews Correlation Coefficient (MCC) ---")
    mcc = matthews_corrcoef(y_true, y_pred)
    print(f"MCC Score: {mcc:.4f}\n")

    return mcc

# =====================================================================
# EJECUCION DE LA COMPARATIVA
# =====================================================================
# Aplicar la transformación
processed_test_dataset = data3["test"].with_transform(transform_images)


# Crear DataLoader
test_loader = DataLoader(
    processed_test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=default_data_collator
)


print("\n" + "#"*60)
print("COMPARATIVA DE RENDIMIENTO: PROFESOR VS ALUMNO")
print("#"*60)

# Rendimiento del Profesor en datos nuevos
mcc_profesor = evaluar_modelo(
    modelo=teacher_model,
    dataloader=test_loader,
    device=device,
    nombre_modelo="Profesor (DINOv2-Large)",
    nombre_dataset="Test"
)

# Rendimiento del Alumno en los mismos datos nuevos
mcc_alumno_test = evaluar_modelo(
    modelo=student_model,
    dataloader=test_loader,
    device=device,
    nombre_modelo="Alumno (DINOv2-Small Destilado)",
    nombre_dataset="Test"
)

# Rendimiento del Alumno en validacion
mcc_alumno_val = evaluar_modelo(
    modelo=student_model,
    dataloader=val_loader,
    device=device,
    nombre_modelo="Alumno (DINOv2-Small Destilado)",
    nombre_dataset="Validacion"
)


############################################################
COMPARATIVA DE RENDIMIENTO: PROFESOR VS ALUMNO
############################################################

EVALUACION: Profesor (DINOv2-Large)
Dataset: Test

--- Reporte de Clasificacion ---
              precision    recall  f1-score   support

           0     1.0000    0.9375    0.9677        16
           1     0.8000    1.0000    0.8889         4

    accuracy                         0.9500        20
   macro avg     0.9000    0.9688    0.9283        20
weighted avg     0.9600    0.9500    0.9520        20

--- Matthews Correlation Coefficient (MCC) ---
MCC Score: 0.8660


EVALUACION: Alumno (DINOv2-Small Destilado)
Dataset: Test

--- Reporte de Clasificacion ---
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        16
           1     1.0000    1.0000    1.0000         4

    accuracy                         1.0000        20
   macro avg     1.0000    1.0000    1.00

In [ ]:
# Apuntamos a la carpeta donde el Trainer guardó el modelo y el procesador
carpeta_base = "./modelo_estudiante_destilado"
nombre_exp = "experimento_dataset3_facebook_dinov2_SMALL_DESTILADO" # El nombre de la carpeta en remoto

archivos_a_subir = []

# Escaneamos la carpeta del estudiante
for raiz, _, ficheros in os.walk(carpeta_base):
    for fichero in ficheros:
        ruta_local = os.path.join(raiz, fichero)
        ruta_relativa = os.path.relpath(ruta_local, carpeta_base)

        ruta_remota = f"Modelos_Entrenados/{nombre_exp}/{ruta_relativa}"

        archivos_a_subir.append((str(ruta_local), ruta_remota))

# Subimos los archivos
try:
    api.batch_bucket_files(
        bucket_id,
        add=archivos_a_subir
    )
    print("Modelo estudiante subido al bucket exitosamente.")
except Exception as e:
    print(f"Error al subir el modelo estudiante al bucket: {e}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../preprocessor_config.json: 100%|##########|   436B /   436B            

  ...nte_destilado/config.json: 100%|##########|   915B /   915B            

  ...stilado/model.safetensors:   1%|1         | 1.11MB / 88.3MB            

Modelo estudiante subido al bucket exitosamente.
